In [2]:
from pybaseball import statcast
from pybaseball import  playerid_lookup
from pybaseball import  statcast_pitcher, batting_stats_range

import pandas as pd

In [4]:
#statcast contains all of the pitch by pitch data
gbg_data = statcast(start_dt='2025-05-01', end_dt='2025-06-30')

# 2. Get unique MLBAM IDs for each role
# It is important to treat them separately even if some players (Ohtani) do both.
unique_pitchers = gbg_data['pitcher'].unique().tolist()
unique_batters = gbg_data['batter'].unique().tolist()

This is a large query, it may take a moment to complete


/opt/anaconda3/envs/testing_env/lib/python3.12/site-packages/pybaseball/statcast.py:50: UserWarning: 
That's a nice request you got there. It'd be a shame if something were to happen to it.
We strongly recommend that you enable caching before running this. It's as simple as `pybaseball.cache.enable()`.
Since the Statcast requests can take a *really* long time to run, if something were to happen, like: a disconnect;
gremlins; computer repair by associates of Rudy Giuliani; electromagnetic interference from metal trash cans; etc.;
you could lose a lot of progress. Enabling caching will allow you to immediately recover all the successful
subqueries if that happens.
  warnings.warn(_OVERSIZE_WARNING)
100%|██████████| 61/61 [00:16<00:00,  3.78it/s]


In [5]:
# Create Pitcher Maps
pitcher_id_to_idx = {id: i for i, id in enumerate(unique_pitchers)}
idx_to_pitcher_id = {i: id for id, i in pitcher_id_to_idx.items()}

# Create Batter Maps
batter_id_to_idx = {id: i for i, id in enumerate(unique_batters)}
idx_to_batter_id = {i: id for id, i in batter_id_to_idx.items()}

print(f"Mapped {len(pitcher_id_to_idx)} pitchers and {len(batter_id_to_idx)} batters.")

Mapped 646 pitchers and 549 batters.


In [7]:
# Map the IDs to their new indices
gbg_data['pitcher_idx'] = gbg_data['pitcher'].map(pitcher_id_to_idx)
gbg_data['batter_idx'] = gbg_data['batter'].map(batter_id_to_idx)

# Verify the first few rows
print(gbg_data[['pitcher', 'pitcher_idx', 'batter', 'batter_idx']].head())

      pitcher  pitcher_idx  batter  batter_idx
1733   621383            0  630105           0
1748   621383            0  630105           0
1813   621383            0  593428           1
1874   621383            0  593428           1
1935   621383            0  593428           1


In [ ]:
print(gbg_data.head)

     pitch_type   game_date  release_speed  release_pos_x  release_pos_z  \
1733         SL  2025-06-30           87.9           2.95           6.34   
1748         SI  2025-06-30           92.7           2.53           6.49   
1813         FF  2025-06-30           93.2           2.59           6.47   
1874         SL  2025-06-30           87.3           2.82           6.42   
1935         SL  2025-06-30           87.9           2.74           6.43   

        player_name  batter  pitcher     events    description  ...  \
1733  Banks, Tanner  630105   621383  field_out  hit_into_play  ...   
1748  Banks, Tanner  630105   621383        NaN           ball  ...   
1813  Banks, Tanner  593428   621383     single  hit_into_play  ...   
1874  Banks, Tanner  593428   621383        NaN           ball  ...   
1935  Banks, Tanner  593428   621383        NaN           ball  ...   

      api_break_x_arm  api_break_x_batter_in  arm_angle  attack_angle  \
1733            -0.56                  -0.5

In [ ]:
import sys
from pathlib import Path

# Ensure imports find `src/` (run this notebook from the project root: MLB_proj)
ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
from torch_geometric.loader import DataLoader

from src.data.build_plate_appearances import build_plate_appearances, DEFAULT_OUTCOME_CLASSES
from src.data.features import RollingTemporalFeatureBuilder
from src.data.graph_dataset import BipartitePAHeteroDataset, GraphDatasetConfig
from src.train import chronological_split, PAOutcomeGNN, make_class_weights, set_seed
from src.eval import evaluate

In [ ]:
# One row per plate appearance; target_class is the AB outcome bucket
pa_df = build_plate_appearances(gbg_data)
pa_df["game_date"] = pd.to_datetime(pa_df["game_date"])
print(f"PA rows: {len(pa_df)}")
print(pa_df["target_class"].value_counts())

In [ ]:
# Chronological split: first 80% of ordered PAs -> train, last 20% -> test
ratio_train = 0.8
set_seed(42)
train_df, test_df = chronological_split(pa_df, ratio=ratio_train)
print(f"train={len(train_df)} | test={len(test_df)}")

classes = [c for c in DEFAULT_OUTCOME_CLASSES if c in set(pa_df["target_class"].unique().tolist())]
label_to_idx = {c: i for i, c in enumerate(classes)}
num_classes = len(classes)
if num_classes < 2:
    raise ValueError(f"Need at least 2 outcome classes; got {classes}")

# Smaller graph settings keep notebook runs tractable; raise toward (50, 30) for full experiments.
rolling_window = 20
neighbor_top_k = 10
feature_builder = RollingTemporalFeatureBuilder(
    pa_df,
    rolling_window=rolling_window,
    outcome_classes=DEFAULT_OUTCOME_CLASSES,
)
dims = feature_builder.get_feature_dims()

cfg = GraphDatasetConfig(rolling_window=rolling_window, neighbor_top_k=neighbor_top_k)
train_ds = BipartitePAHeteroDataset(train_df, feature_builder=feature_builder, label_to_idx=label_to_idx, config=cfg)
test_ds = BipartitePAHeteroDataset(test_df, feature_builder=feature_builder, label_to_idx=label_to_idx, config=cfg)

# Hetero batching: use 1 until the encoder supports variable graph sizes in a batched batch.
batch_size = 1
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = PAOutcomeGNN(
    node_in_dim=dims.node_dim,
    edge_attr_dim=dims.edge_dim,
    hidden_dim=64,
    num_layers=2,
    num_heads=4,
    num_classes=num_classes,
    dropout=0.1,
    mlp_hidden_dim=128,
).to(device)

# Cross-entropy on AB outcome class (weighted by inverse frequency on the train slice)
class_weights = make_class_weights(train_df, classes, label_to_idx, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 2  # increase for real training
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss, n_batches = 0.0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad(set_to_none=True)
        logits = model(batch)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        epoch_loss += float(loss.detach().cpu().item())
        n_batches += 1

    metrics = evaluate(model, test_loader, device=device, num_classes=num_classes)
    print(
        f"Epoch {epoch:02d} | train_loss={epoch_loss / max(n_batches, 1):.4f} "
        f"| test_acc={metrics.accuracy:.4f} | test_macro_f1={metrics.macro_f1:.4f} "
        f"| test_logloss={metrics.logloss:.4f}"
    )